# Planograma OXXO — Backend (Mueble BCO / CF) — Factibilidad horizontal

**Cambio de regla de negocio respecto a la versión anterior:** se confirmó que los
productos altos (ej. refrescos familiares) siempre van en la charola inferior, que
tiene holgura vertical de sobra. Por lo tanto, el modelo matemático **omite
intencionalmente** las restricciones de `ALTO` y `PROFUNDO`.

La heurística ahora valida únicamente:

1. **Capacidad horizontal:** `ancho_usado + ancho_producto <= WIDTH_MAX`.
2. **Variedad por charola:** máximo `MAX_SKUS_POR_CHAROLA = 10` SKUs distintos por
   charola.

La altura de la charola (`EYE_LEVEL_SHELVES`) deja de ser una restricción y pasa a
ser **únicamente un bono en la función objetivo de la Fase 3 (2-opt)**, premiando
productos de alta demanda colocados en charolas "a la altura de los ojos".

Sin `matplotlib` — el dibujo lo hace Streamlit a partir de `output_planograma.csv`.

In [3]:
# ============================================================
# 1) IMPORTS Y CARGA
# ============================================================
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv('Ejemplo.csv')
df.columns = [str(c).strip() for c in df.columns]
print(f"Filas originales: {len(df):,}")


Filas originales: 497,437


In [ ]:
# ============================================================
# 2) CONFIGURACION
# ============================================================
MUEBLE_FILTRO         = 'CF'    # se aplica a todos los segmentos

WIDTH_MAX             = 120.0   # ancho maximo por charola (cm)
MAX_SKUS_POR_CHAROLA  = 10      # limite de variedad por charola
EYE_LEVEL_SHELVES     = {1, 2}  # 0-based; solo afecta el objetivo (Fase 3)

for col in ('SEGMENTO_ID', 'MUEBLE_ID'):
    if col not in df.columns:
        raise KeyError(f"La columna obligatoria '{col}' no existe en el CSV.")

df_mueble = df[df['MUEBLE_ID'] == MUEBLE_FILTRO].copy()
del df

if df_mueble.empty:
    raise ValueError(f"No hay filas con MUEBLE_ID='{MUEBLE_FILTRO}'.")

segmentos = sorted(df_mueble['SEGMENTO_ID'].dropna().unique().tolist())
print(f"Mueble: {MUEBLE_FILTRO} | Segmentos encontrados ({len(segmentos)}): {segmentos}")


## Función `procesar_segmento`

Encapsula las fases 1-3 (limpieza, FFD, bin-packing, 2-opt) y la exportación a CSV
para **un segmento dado**. Se llama en el loop de la celda siguiente.

In [ ]:
# ============================================================
# 3) CLASE CHAROLA + FUNCIONES DE FASE 1-3
#    Cada charola tiene dos zonas:
#      - Zona DI (derecha a izquierda): se llena desde el borde derecho
#      - Zona ID (izquierda a derecha): se llena desde el borde izquierdo
# ============================================================

class Charola:
    __slots__ = ('idx', 'width_max', 'ancho_ltr', 'ancho_rtl', 'productos')

    def __init__(self, idx, width_max):
        self.idx       = idx
        self.width_max = width_max
        self.ancho_ltr = 0.0   # espacio acumulado desde la izquierda (ID)
        self.ancho_rtl = 0.0   # espacio acumulado desde la derecha  (DI)
        self.productos = []

    @property
    def ancho_usado(self):
        return self.ancho_ltr + self.ancho_rtl

    def cabe(self, ancho_total):
        return (self.ancho_ltr + self.ancho_rtl + ancho_total <= self.width_max and
                len(self.productos) < MAX_SKUS_POR_CHAROLA)

    def agregar(self, p):
        if p['direccion'] == 'DI':
            self.ancho_rtl  += p['ancho_total_cm']
            p['x_inicio']    = self.width_max - self.ancho_rtl
        else:                                           # ID
            p['x_inicio']    = self.ancho_ltr
            self.ancho_ltr  += p['ancho_total_cm']
        self.productos.append(p)

    def pct_ocupacion(self):
        return self.ancho_usado / self.width_max * 100 if self.width_max else 0.0

    def recalcular_posiciones(self):
        """Recalcula x_inicio tras un swap en 2-opt."""
        ltr, rtl = 0.0, 0.0
        # Acumular RTL para saber el total y posicionar de derecha a izquierda
        rtl_total = sum(p['ancho_total_cm'] for p in self.productos if p['direccion'] == 'DI')
        cur_rtl = self.width_max - rtl_total
        for p in self.productos:
            if p['direccion'] == 'DI':
                p['x_inicio'] = cur_rtl
                cur_rtl += p['ancho_total_cm']
            else:
                p['x_inicio'] = ltr
                ltr += p['ancho_total_cm']
        self.ancho_ltr = ltr
        self.ancho_rtl = rtl_total


def _limpiar(df_seg):
    cols_base = ['SEGMENTO_ID', 'MUEBLE_ID', 'PLANOGRUPO_DESC', 'TAMANO_DESC',
                 'DIRECCION_LEGO_ID', 'UPC_CVE', 'NUM_FRENTES', 'CHAROLA',
                 'UBICACION_BANDEJA', 'ANCHO', 'ITEM_DESC']
    cols_dedup = [c for c in cols_base if c in df_seg.columns]
    df_seg = df_seg.drop_duplicates(subset=cols_dedup).copy()
    for c in ['ANCHO', 'NUM_FRENTES']:
        df_seg[c] = pd.to_numeric(df_seg[c], errors='coerce')
    df_seg = df_seg.dropna(subset=['ANCHO'])
    df_seg = df_seg[df_seg['ANCHO'] > 0]
    df_seg['NUM_FRENTES'] = df_seg['NUM_FRENTES'].fillna(1).clip(lower=1)
    if 'DIRECCION_LEGO_ID' not in df_seg.columns:
        df_seg['DIRECCION_LEGO_ID'] = 'ID'
    df_seg['DIRECCION_LEGO_ID'] = df_seg['DIRECCION_LEGO_ID'].fillna('ID')

    id_col = 'UPC_CVE' if ('UPC_CVE' in df_seg.columns and df_seg['UPC_CVE'].notna().any()) else 'ITEM_DESC'
    agg_dict = {
        'ITEM_DESC':          'first',
        'PLANOGRUPO_DESC':    'first',
        'ANCHO':              'first',
        'NUM_FRENTES':        'median',
        'DIRECCION_LEGO_ID':  'first',
    }
    if id_col != 'UPC_CVE' and 'UPC_CVE' in df_seg.columns:
        agg_dict['UPC_CVE'] = 'first'
    df_seg = df_seg.groupby(id_col, as_index=False).agg(agg_dict)
    return df_seg, id_col


def _ordenar_ffd(df_seg):
    df_seg = df_seg[df_seg['ANCHO'] <= WIDTH_MAX].copy()
    cap_frentes = (WIDTH_MAX // df_seg['ANCHO']).astype(int).clip(lower=1)
    df_seg['NUM_FRENTES']     = np.minimum(df_seg['NUM_FRENTES'].astype(int), cap_frentes)
    df_seg['ancho_total_cm']  = df_seg['ANCHO'] * df_seg['NUM_FRENTES']
    df_seg['score_demanda']   = df_seg['NUM_FRENTES']
    df_seg['PLANOGRUPO_DESC'] = df_seg['PLANOGRUPO_DESC'].fillna("SIN_CATEGORIA").astype(str)
    df_seg['ITEM_DESC']       = df_seg['ITEM_DESC'].fillna("SIN_NOMBRE").astype(str)
    # FFD por separado dentro de cada grupo de direccion
    return df_seg.sort_values(
        ['DIRECCION_LEGO_ID', 'ancho_total_cm'],
        ascending=[True, False]          # DI antes que ID (D < I alfabeticamente)
    ).reset_index(drop=True)


def _empacar(df_sorted):
    """
    Empaca primero los productos DI (desde la derecha) y luego los ID (desde la
    izquierda). Ambos grupos comparten las mismas charolas.
    """
    charolas = []

    def _pack_grupo(sub):
        for row in sub.itertuples(index=False):
            p = {
                'UPC_CVE':         getattr(row, 'UPC_CVE', None),
                'ITEM_DESC':       row.ITEM_DESC,
                'PLANOGRUPO_DESC': row.PLANOGRUPO_DESC,
                'direccion':       row.DIRECCION_LEGO_ID,
                'ANCHO':           float(row.ANCHO),
                'NUM_FRENTES':     int(row.NUM_FRENTES),
                'ancho_total_cm':  float(row.ancho_total_cm),
                'score_demanda':   int(row.score_demanda),
            }
            colocado = False
            for c in charolas:
                if c.cabe(p['ancho_total_cm']):
                    c.agregar(p)
                    colocado = True
                    break
            if not colocado:
                c = Charola(len(charolas), WIDTH_MAX)
                c.agregar(p)
                charolas.append(c)

    # Primero DI (derecha a izquierda), luego ID (izquierda a derecha)
    _pack_grupo(df_sorted[df_sorted['DIRECCION_LEGO_ID'] == 'DI'])
    _pack_grupo(df_sorted[df_sorted['DIRECCION_LEGO_ID'] != 'DI'])
    return charolas


def _calcular_objetivo(charolas):
    return sum(
        p['score_demanda'] * (1.5 if c.idx in EYE_LEVEL_SHELVES else 1.0)
        for c in charolas for p in c.productos
    )


def _2opt(charolas, max_iter=300):
    """Solo intercambia productos de la misma direccion para no mezclar zonas."""
    n_mejoras = 0
    for _ in range(max_iter):
        mejor_delta, mejor_swap = 0.0, None
        for i in range(len(charolas)):
            for j in range(i + 1, len(charolas)):
                ca, cb = charolas[i], charolas[j]
                fa = 1.5 if ca.idx in EYE_LEVEL_SHELVES else 1.0
                fb = 1.5 if cb.idx in EYE_LEVEL_SHELVES else 1.0
                for ia, pa in enumerate(ca.productos):
                    for ib, pb in enumerate(cb.productos):
                        if pa['direccion'] != pb['direccion']:
                            continue   # no mezclar DI con ID
                        dir_ = pa['direccion']
                        if dir_ == 'DI':
                            na_ltr, na_rtl = ca.ancho_ltr, ca.ancho_rtl - pa['ancho_total_cm'] + pb['ancho_total_cm']
                            nb_ltr, nb_rtl = cb.ancho_ltr, cb.ancho_rtl - pb['ancho_total_cm'] + pa['ancho_total_cm']
                        else:
                            na_ltr, na_rtl = ca.ancho_ltr - pa['ancho_total_cm'] + pb['ancho_total_cm'], ca.ancho_rtl
                            nb_ltr, nb_rtl = cb.ancho_ltr - pb['ancho_total_cm'] + pa['ancho_total_cm'], cb.ancho_rtl
                        if na_ltr + na_rtl > ca.width_max or nb_ltr + nb_rtl > cb.width_max:
                            continue
                        delta = (pb['score_demanda'] - pa['score_demanda']) * (fa - fb)
                        if delta > mejor_delta + 1e-9:
                            mejor_delta = delta
                            mejor_swap  = (i, ia, j, ib, na_ltr, na_rtl, nb_ltr, nb_rtl)
        if mejor_swap is None:
            break
        i, ia, j, ib, na_ltr, na_rtl, nb_ltr, nb_rtl = mejor_swap
        ca, cb = charolas[i], charolas[j]
        ca.productos[ia], cb.productos[ib] = cb.productos[ib], ca.productos[ia]
        ca.ancho_ltr, ca.ancho_rtl = na_ltr, na_rtl
        cb.ancho_ltr, cb.ancho_rtl = nb_ltr, nb_rtl
        n_mejoras += 1

    for c in charolas:
        c.recalcular_posiciones()
    return charolas, n_mejoras


def procesar_segmento(df_mueble, segmento, mueble, carpeta_salida='.'):
    """Pipeline completo para un segmento: limpieza, FFD, bin-packing, 2-opt y export CSV."""
    df_seg = df_mueble[df_mueble['SEGMENTO_ID'] == segmento].copy()
    if df_seg.empty:
        print(f"  [SKIP] Segmento {segmento}: sin datos.")
        return None

    df_seg, _ = _limpiar(df_seg)
    df_work   = _ordenar_ffd(df_seg)
    charolas  = _empacar(df_work)

    z_antes = _calcular_objetivo(charolas)
    charolas, n_mejoras = _2opt(charolas)
    z_despues = _calcular_objetivo(charolas)

    ocup_prom = np.mean([c.pct_ocupacion() for c in charolas]) if charolas else 0.0
    n_di = (df_work['DIRECCION_LEGO_ID'] == 'DI').sum()
    n_id = (df_work['DIRECCION_LEGO_ID'] != 'DI').sum()
    print(f"  Segmento {segmento:>6} | DI={n_di:>3} ID={n_id:>3} | "
          f"Charolas={len(charolas):>2} | Ocup.prom={ocup_prom:5.1f}% | "
          f"Obj {z_antes:.0f}->{z_despues:.0f} | Swaps={n_mejoras}")

    registros = [
        {
            'SEGMENTO_ID':         segmento,
            'MUEBLE_ID':           mueble,
            'CHAROLA_ID':          c.idx + 1,
            'UPC_CVE':             p['UPC_CVE'],
            'ITEM_DESC':           p['ITEM_DESC'],
            'PLANOGRUPO_DESC':     p['PLANOGRUPO_DESC'],
            'DIRECCION':           p['direccion'],
            'COORDENADA_X_INICIO': round(p['x_inicio'], 2),
            'ANCHO_DIBUJO_CM':     round(p['ancho_total_cm'], 2),
        }
        for c in charolas for p in c.productos
    ]
    df_out = pd.DataFrame(registros)
    df_out.to_csv(f"{carpeta_salida}/output_planograma_{segmento}.csv", index=False)
    return df_out

print("Funciones cargadas.")


In [ ]:
# ============================================================
# 4) LOOP POR SEGMENTO
#    Genera: output_planograma_{SEGMENTO}.csv  (uno por segmento)
#            output_planograma_todos.csv       (consolidado)
# ============================================================
CARPETA_SALIDA = '.'   # cambia si quieres guardar en otra ruta

print(f"Procesando {len(segmentos)} segmentos para MUEBLE={MUEBLE_FILTRO}...\n")

resultados = {}
dfs_todos  = []

for seg in segmentos:
    df_result = procesar_segmento(df_mueble, seg, MUEBLE_FILTRO, CARPETA_SALIDA)
    if df_result is not None:
        resultados[seg] = df_result
        dfs_todos.append(df_result)

if dfs_todos:
    df_consolidado = pd.concat(dfs_todos, ignore_index=True)
    ruta = f"{CARPETA_SALIDA}/output_planograma_todos.csv"
    df_consolidado.to_csv(ruta, index=False)
    print(f"\nConsolidado: {ruta} ({len(df_consolidado)} filas)")

print(f"\nSegmentos procesados: {list(resultados.keys())}")
